In [586]:
from dotenv import load_dotenv
load_dotenv()

True

In [587]:
from langchain_groq import ChatGroq

llm1 = ChatGroq(
    model="llama-3.3-70b-versatile",   
    temperature=0,
)

llm2 = ChatGroq(
    model="llama-3.1-8b-instant",   
    temperature=0,
)

In [171]:
from langchain.tools import tool

@tool 
def add(a:int,b:int) -> str:
    '''
    Use this tool to add numbers
    '''
    return a+b

In [172]:
from langchain.tools import tool

@tool 
def add(a:int,b:int) -> str:
    '''
    Use this tool to add numbers
    '''
    return a+b
@tool 
def sub(a:int,b:int) -> str:
    '''
    Use this tool to subtraction numbers
    '''
    return a-b

In [20]:
def get_tools(mode: str):
    if mode == "add_only":
        return [add]

    elif mode == "subtract_only":
        return [sub]

    elif mode == "both":
        return [add, sub]


In [588]:
from langchain.agents import create_agent

prompt1='You are an Math Assistant'
prompt2='You are an Biology Assistant'

In [589]:

main_agent=create_agent(model=llm1,system_prompt=prompt2)

In [221]:
from langchain.messages import HumanMessage

response = main_agent.invoke({
    "messages": HumanMessage(content="hi")
})


In [590]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt, ModelRequest,ModelResponse,wrap_model_call,wrap_tool_call
from typing import Callable

@wrap_model_call
def user_language_prompt(request: ModelRequest,handler: Callable[[ModelRequest],ModelResponse]) -> str:
    """select model based on user request for """
    query = request.messages[0].content
    list1=['+','-','*','hello']
    for word in list1:
        if word in query:
            request=request.override(system_prompt=prompt1)
        else:
            request=request.override(system_prompt=prompt2)
    print(len(query))
    return handler(request)

In [591]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm1,
    middleware=[user_language_prompt],
    system_prompt=prompt2,
)

In [594]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="hi")]},
)

print(response["messages"][-1].content)

2
Hello. I'm a Biology Assistant. How can I help you today? Do you have a question about a specific topic in biology, or would you like to learn about a particular subject?


In [189]:
response["messages"]

[HumanMessage(content='2+3', additional_kwargs={}, response_metadata={}, id='5b43f466-ac30-4b75-8160-2e6be7b08e6c'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'jxqqkpxb9', 'function': {'arguments': '{"a":2,"b":3}', 'name': 'add'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 226, 'total_tokens': 244, 'completion_time': 0.036977031, 'completion_tokens_details': None, 'prompt_time': 0.011028848, 'prompt_tokens_details': None, 'queue_time': 0.058677714, 'total_time': 0.048005879}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c080c-6ffc-7ee1-adac-78d674b6033d-0', tool_calls=[{'name': 'add', 'args': {'a': 2, 'b': 3}, 'id': 'jxqqkpxb9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 226, 'output_tokens': 18, 'total_tokens': 244}),
 T

In [640]:
from langchain.tools import tool

@tool 
def add(a:int,b:int) -> str:
    '''
    Use this tool to add numbers
    '''
    return a-b
@tool 
def sub(a:int,b:int) -> str:
    '''
    Use this tool to subtraction numbers
    '''
    return a+b

In [641]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""
    tools = [add] 
    request = request.override(tools=tools) 

    return handler(request)


In [613]:
agent1 = create_agent(
    model=llm1,
    # middleware=[dynamic_tool_call],
    system_prompt='''
    You need to answer user query based on the tools you have , Use tools to answer the queries
    ''',
    tools=[sub]
)

In [614]:
from langchain.messages import HumanMessage

response = agent1.invoke(
    {"messages": [HumanMessage(content="add 2 and 3")]},
)

# print(response["messages"][-1].content)

In [615]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='add 2 and 3', additional_kwargs={}, response_metadata={}, id='f326b2f4-8b70-4502-ae1a-84a713e05410'),
              AIMessage(content='I am not able to execute this task as it exceeds the limitations of the functions I have been given.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 243, 'total_tokens': 265, 'completion_time': 0.176706348, 'completion_tokens_details': None, 'prompt_time': 0.050506473, 'prompt_tokens_details': None, 'queue_time': 0.059654617, 'total_time': 0.227212821}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c091f-ef75-7bd1-badb-44f4b29279d8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 243, 'output_tokens': 22, 'total_tokens': 265})]}


In [617]:
from langchain.messages import ToolMessage

for msg in response["messages"]:
    if isinstance(msg, ToolMessage):
        print("Tool:", msg.name)
        print("Result:", msg.content)


This is for dynamic for tools

In [631]:
from langchain.tools import tool

@tool 
def add(a:int,b:int) -> str:
    '''
    Use this tool to subtraction numbers
    '''
    return a-b
@tool 
def sub(a:int,b:int) -> str:
    '''
    Use this tool to add numbers
    '''
    return a+b



In [642]:
from langchain.agents.middleware import wrap_model_call
@wrap_model_call
def dynamic_tool_injector(request, handler):
    query = request.messages[-1].content.lower()
    tools = []
    if "add" in query or "+" in query:
        # print("hello") proof such that this is entering to this block
        tools.append(add)

    if "sub" in query or "-" in query:
        # print("hi") proof such that this is entering to this block
        tools.append(sub)

    if tools:
        request = request.override(tools=tools)

    return handler(request)

In [643]:
from langchain.agents.middleware import wrap_model_call

@wrap_model_call
def debug_tools(request, handler):
    print("=== MODEL CALL DEBUG ===")
    print("TOOLS SENT TO MODEL:", [t.name for t in (request.tools or [])])
    print("SYSTEM PROMPT:", request.system_message.content)
    print("USER MESSAGE:", request.messages[-1].content)
    print("========================\n")

    return handler(request)


In [647]:

agent1 = create_agent(
    model=llm1,
    middleware=[dynamic_tool_injector,debug_tools],
    system_prompt="""
    You are a math assistant.
    To perform some operation use the tools before answering 
    and also run the dynmaic_tool_injector to get new tools
    """,
    tools=[sub,add] #mention all details of it
)


In [648]:
from langchain.messages import HumanMessage

response = agent1.invoke(
    {"messages": [HumanMessage(content="add 2 and 3")]}
)

print(response["messages"][-1].content)


=== MODEL CALL DEBUG ===
TOOLS SENT TO MODEL: ['add']
SYSTEM PROMPT: 
    You are a math assistant.
    To perform some operation use the tools before answering 
    and also run the dynmaic_tool_injector to get new tools
    
USER MESSAGE: add 2 and 3

=== MODEL CALL DEBUG ===
TOOLS SENT TO MODEL: ['sub']
SYSTEM PROMPT: 
    You are a math assistant.
    To perform some operation use the tools before answering 
    and also run the dynmaic_tool_injector to get new tools
    
USER MESSAGE: -1

=== MODEL CALL DEBUG ===
TOOLS SENT TO MODEL: ['sub', 'add']
SYSTEM PROMPT: 
    You are a math assistant.
    To perform some operation use the tools before answering 
    and also run the dynmaic_tool_injector to get new tools
    
USER MESSAGE: 8

The result of adding 2 and 3 is 5.


In [639]:
response["messages"][-1].content

'The result of subbing 2 and 3 is 5.'

In [576]:
@wrap_model_call
def debug(request, handler):
    print("TOOLS SENT TO MODEL:", [t.name for t in request.tools or []])
    return handler(request)
